# 🔄 Chapter 8: The Matrix Inverse

---

## 8.1 Introduction: What is a Matrix Inverse?

In basic arithmetic, every non-zero number has a reciprocal (or inverse). For example, the inverse of $5$ is $\frac{1}{5}$, and multiplying them together yields $1$. This allows us to "undo" multiplication. If we have the equation $5x = 20$, we multiply both sides by $\frac{1}{5}$ to find $x = 4$.

In linear algebra, we frequently encounter systems of equations written in matrix form:
$$\mathbf{A}\mathbf{x} = \mathbf{b}$$

We want to isolate the vector $\mathbf{x}$. However, **there is no such thing as matrix division**. We cannot simply do $\mathbf{x} = \mathbf{b} / \mathbf{A}$. Instead, we use the **Matrix Inverse**, denoted as $\mathbf{A}^{-1}$. 

If we multiply a matrix by its inverse, we get the Identity Matrix $\mathbf{I}$ (the matrix equivalent of the number 1):
$$\mathbf{A}^{-1} \mathbf{A} = \mathbf{I}$$

Therefore, to solve for $\mathbf{x}$, we multiply both sides by $\mathbf{A}^{-1}$:
$$\mathbf{A}^{-1} \mathbf{A} \mathbf{x} = \mathbf{A}^{-1} \mathbf{b} \implies \mathbf{I}\mathbf{x} = \mathbf{A}^{-1} \mathbf{b} \implies \mathbf{x} = \mathbf{A}^{-1} \mathbf{b}$$

In this chapter, we will explore how to compute the inverse, the strict conditions required for a matrix to have an inverse, and what to do when those conditions are not met.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 8.")

## 8.2 Computing the Inverse

For a matrix to have a standard inverse, it must meet two strict conditions:
1. It must be a **Square Matrix** ($M = N$).
2. It must be **Full Rank** (all of its columns must be linearly independent, meaning its determinant is not zero).

Let's create a full-rank square matrix and find its inverse using NumPy's `np.linalg.inv()` function.

In [ ]:
# Create a 3x3 square matrix
A = np.array([
    [2, 1, 3],
    [0, 5, 2],
    [4, 2, 1]
])

# Compute the inverse
A_inv = np.linalg.inv(A)

print("Original Matrix A:")
print(A)

print("\nInverse Matrix A^(-1):")
print(np.round(A_inv, 3))

# Let's verify that A @ A_inv equals the Identity Matrix (I)
identity_test = A @ A_inv
print("\nVerification (A @ A_inv):")
print(np.round(identity_test, 3))
print("Notice the 1s on the diagonal and 0s elsewhere. It is the Identity Matrix!")

## 8.3 Singular Matrices (The Un-invertible)

What happens if a matrix is **not** full rank? Such a matrix is called **Singular**.

Geometrically, if a 3D matrix is rank-deficient (e.g., Rank 2), it means the matrix collapses 3D space into a flat 2D plane. Once space is squashed flat, multiple points map to the exact same location. You cannot "un-squash" the space to find where a specific point originally came from. Therefore, the transformation cannot be undone.

Algebraically, attempting to invert a singular matrix is equivalent to trying to **divide by zero**. The determinant of a singular matrix is $0$, and the formula for an inverse involves dividing by the determinant.

In [ ]:
# Create a Rank-Deficient (Singular) Matrix
# We will make the 3rd column a direct sum of the 1st and 2nd columns.
S = np.array([
    [1, 2, 3],  # 1 + 2 = 3
    [4, 5, 9],  # 4 + 5 = 9
    [7, 8, 15]  # 7 + 8 = 15
])

print(f"Matrix S rank: {np.linalg.matrix_rank(S)} (It is a 3x3 matrix, but rank is only 2)")

try:
    # Attempting to invert a singular matrix will throw a LinAlgError
    S_inv = np.linalg.inv(S)
except np.linalg.LinAlgError as e:
    print(f"\nERROR: {e}")
    print("You cannot invert a singular (rank-deficient) matrix because it is like dividing by zero!")

## 8.4 The Moore-Penrose Pseudoinverse

In real-world data science, matrices are rarely perfectly square, and they are frequently rank-deficient due to collinear features. 

To handle this, we use the **Pseudoinverse** (specifically the Moore-Penrose Pseudoinverse, denoted as $\mathbf{A}^+$). 
- If the matrix is square and full rank, the pseudoinverse returns the exact standard inverse.
- If the matrix is rectangular or singular, the pseudoinverse returns the "best possible approximation" of an inverse that minimizes the squared error.

The pseudoinverse is the algorithmic workhorse behind Ordinary Least Squares (OLS) Linear Regression.

In [ ]:
# 1. Applying pseudoinverse to a Rectangular Matrix (4x2)
R = np.random.randint(1, 10, size=(4, 2))
R_pinv = np.linalg.pinv(R)

print("Rectangular Matrix R (4x2):")
print(R)
print("\nPseudoinverse R^+ (2x4):")
print(np.round(R_pinv, 3))

# 2. Applying pseudoinverse to the Singular Matrix from the previous cell
S_pinv = np.linalg.pinv(S)
print("\nRemember Matrix S? The one that threw an error because it was singular?")
print("Using np.linalg.pinv() works flawlessly:")
print(np.round(S_pinv, 3))

## 8.5 Numerical Stability and Floating-Point Artifacts

A vital lesson in applied linear algebra is that computers do not calculate math symbolically; they use floating-point numbers. 

Because computers have finite memory (usually 64-bit precision), calculating the inverse can introduce tiny rounding errors. When you multiply a matrix by its inverse on a computer, you might not get *exactly* zero in the off-diagonal elements. Instead, you might see extremely small numbers like `1.4e-16` (which means $1.4 \times 10^{-16}$ or $0.00000000000000014$).

This is called a **floating-point artifact**. In Data Science, you must always be prepared to round these numbers or use tolerance thresholds (`np.isclose`) rather than checking for exact equality (`==`).

In [ ]:
# Create a seemingly simple matrix
M = np.array([
    [4, 7],
    [2, 6]
])

M_inv = np.linalg.inv(M)
identity_computed = M @ M_inv

print("Computed Identity Matrix (Without rounding):")
print(identity_computed)

print("\nNotice the value at [1, 0]:", identity_computed[1, 0])
print("It is not 0.0, but rather a tiny microscopic number due to computer precision limits.")

# Best practice for comparing matrices in Python:
true_identity = np.eye(2)
is_exact = np.array_equal(identity_computed, true_identity)
is_close = np.allclose(identity_computed, true_identity)

print(f"\nUsing exact equality (==)   : {is_exact}  <-- Dangerous in Linear Algebra!")
print(f"Using np.allclose (tolerance) : {is_close}  <-- Best Practice!")

## 8.6 Chapter Summary

In Chapter 8, we explored the Matrix Inverse:
- **The Concept:** The inverse is the matrix equivalent of dividing, allowing us to isolate variables in algebraic equations.
- **The Conditions:** A true inverse only exists for square, full-rank matrices. Singular (rank-deficient) matrices collapse space and cannot be inverted because information is lost.
- **The Pseudoinverse:** The Moore-Penrose pseudoinverse ($A^+$) provides a robust fallback for singular and rectangular matrices, making it indispensable for real-world datasets.
- **Numerical Stability:** Be cautious of floating-point artifacts when relying on computer algorithms to compute inverses. Always use tolerance functions like `np.allclose()`.

While computing the inverse is mathematically elegant, in deep learning and massive datasets, directly computing $A^{-1}$ is often too slow and computationally expensive. In practice, algorithms use indirect optimization methods (like Gradient Descent or Matrix Decompositions) which we will explore in the upcoming chapters.